# simpson_paradox

> 对应代码：`EconometricsCode/code_in_notes/Chap.8/simpson_paradox.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.8`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.8")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们模拟 1000 个个体，并生成性别虚拟变量。

In [ ]:
%%stata
clear
set obs 1000
gen sex=runiform()<0.5

我们让锻炼习惯（exer）的分布依赖于性别：男性锻炼概率 0.8，女性仅 0.3。这样性别同时影响锻炼和结果变量，成为一个混杂因素。

In [ ]:
%%stata
gen     exer=runiform()<0.8 if sex==1
replace exer=runiform()<0.3 if sex==0

我们生成结果变量 y：其真实模型中锻炼的效应为 3，性别效应为 -10，并附加标准正态噪声。锻炼的真实效应是正的，但锻炼与性别正相关、性别效应为负，混杂会让全样本估计失真。

In [ ]:
%%stata
gen y=80-10*sex+3*exer+rnormal()

我们先在合起来的全样本中回归 y 对 exer。由于遗漏了性别这一混杂因素，exer 的系数吸收了性别差异，甚至可能呈现虚假的负相关。

In [ ]:
%%stata
reg y exer
outreg2 using simpson_paradox.tex, replace ctitle(全样本)

接着我们分性别分别回归。在每组内部，锻炼与性别不再相关，系数应接近真实效应 3。

In [ ]:
%%stata
reg y exer if sex==1
outreg2 using simpson_paradox.tex, append ctitle(男性)
reg y exer if sex==0
outreg2 using simpson_paradox.tex, append ctitle(女性)

最后我们在全样本回归中加入性别控制变量。控制混杂因素后，exer 的系数同样恢复到 3 附近。这正是辛普森悖论：合并数据中的关系方向可以与分组数据相反，而加入适当的控制变量即可消除混杂、恢复真实效应。

In [ ]:
%%stata
reg y exer sex
outreg2 using simpson_paradox.tex, append ctitle(全样本)